In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
import uproot
import glob

In [ ]:
%pip install uproot

In [ ]:
classes = ['QCD', 'Hbb', 'Hcc', 'Hgg', 'H4q', 'Hqql', 'Zqq', 'Wqq', 'Tbqq', 'Tbl']
n_classes = len(classes)
label_list = [f'label_{cls}' for cls in classes]
score_list = [f'score_label_{cls}' for cls in classes]


RUNS = {
    'ParT (baseline)': '/ParTGMP/runs/part_2m_seed0/pred.root',
    'ParT + GMP (single scale)': [
        '/ParTGMP/runs/partgmp_2m_seed0/pred.root',
        '/ParTGMP/runs/partgmp_2m_seed1/pred.root',
        '/ParTGMP/runs/partgmp_2m_seed2/pred.root',
    ],
    'ParT + GMP (multi scale)': [
        '/ParTGMP/runs/partgmp_multiscale_2m_seed0/pred.root',
        '/ParTGMP/runs/partgmp_multiscale_2m_seed1/pred.root',
    ],
}

In [ ]:
def load_pred(path):
    """Load pred ROOT file(s) matching a glob. Returns (labels [N,10], y_prob [N,10])."""
    arrays = []
    for fname in glob.glob(path):
        with uproot.open(fname) as f:
            arrays.append(f['Events'].arrays(label_list + score_list))
    if not arrays:
        raise FileNotFoundError(f'No files found: {path}')
    concat = {k: np.concatenate([a[k].to_numpy() for a in arrays]) for k in label_list + score_list}
    y_prob = np.stack([concat[k] for k in score_list], axis=1)
    labels = np.stack([concat[k] for k in label_list], axis=1).astype(int)
    return labels, y_prob


def compute_rejections(labels, y_prob):
    """Compute AUC, accuracy, and per-class background rejections."""
    overall_roc_auc = roc_auc_score(labels, y_prob, average='macro', multi_class='ovo')
    predicted_labels = np.argmax(y_prob, axis=1)
    true_labels = np.argmax(labels, axis=1)
    accuracy = accuracy_score(true_labels, predicted_labels)

    scores = y_prob / (y_prob[:, :1] + y_prob)  # p_sig / (p_QCD + p_sig)

    results = {'auc': overall_roc_auc, 'acc': accuracy}

    for i in range(1, n_classes):
        if i == 5:
            percent = 0.99
        elif i == 9:
            percent = 0.995
        else:
            percent = 0.5

        mask = (labels[:, 0] == 1) | (labels[:, i] == 1)
        binary_labels = (labels[mask][:, i] == 1).astype(int)
        binary_scores = scores[mask][:, i]

        fpr, tpr, _ = roc_curve(binary_labels, binary_scores)

        # first idx where tpr >= target (safer than argmin)
        valid = np.where(tpr >= percent)[0]
        if len(valid) == 0:
            rejection = np.nan
        else:
            idx = valid[0]
            rejection = 1 / fpr[idx] if fpr[idx] != 0 else np.inf

        results[classes[i]] = rejection

    return results


def evaluate_run(paths):
    """
    paths: str (single run) or list of str (multi-seed).
    Returns (means_dict, stds_dict). stds_dict is None for single runs.
    """
    if isinstance(paths, str):
        paths = [paths]

    seed_results = []
    for path in paths:
        try:
            labels, y_prob = load_pred(path)
            seed_results.append(compute_rejections(labels, y_prob))
        except FileNotFoundError as e:
            print(f'  WARNING: {e} -- skipping')

    if not seed_results:
        return None, None

    keys = seed_results[0].keys()
    means = {k: np.nanmean([r[k] for r in seed_results]) for k in keys}
    # sample std (ddof=1), only meaningful with >1 seed
    stds = {k: np.nanstd([r[k] for r in seed_results], ddof=1) for k in keys} if len(seed_results) > 1 else None
    return means, stds

In [ ]:
for run_name, paths in RUNS.items():
    print(f'\n=== {run_name} ===')
    means, stds = evaluate_run(paths)
    if means is None:
        print('  No valid predictions found.')
        continue

    n_seeds = len(paths) if isinstance(paths, list) else 1
    seed_str = f'{n_seeds} seed{"s" if n_seeds > 1 else ""}'

    def fmt(key, is_float=False):
        v = means[key]
        s = stds[key] if stds else None
        if np.isinf(v):
            return 'inf'
        if is_float:
            return f'{v:.4f}' + (f' ± {s:.4f}' if s is not None else '')
        return f'{int(round(v))}' + (f' ± {int(round(s))}' if s is not None else '')

    print(f'  [{seed_str}] ROC AUC = {fmt("auc", True)}, Accuracy = {fmt("acc", True)}')
    for i in range(1, n_classes):
        cls = classes[i]
        pct = 'Rej99%' if i == 5 else 'Rej99.5%' if i == 9 else 'Rej50%'
        print(f'  Rejection at {pct} for label_{cls}: {fmt(cls)}')